# Conversion: team_failures

project.convert made this notebook from `original.hql`, beside it; `scan.md` says what it found, line by line. Its logic cell builds the same query from the Toolbox's own functions: `converted()` gives the Statement. Check each `# CONVERSION:` mark, and make each `unchecked_` piece a Toolbox function where you can. The cells below compare it with the original: its Hive, then its rows, through your query. When it gives the rows you want, finish it from `admin/convert.ipynb`: `project.finish_conversion("team_failures", table="<db>.<table>")`.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import compare_hive, compare_with_original, show_hive
from warehouse import query

In [ ]:
# logic
from sqlglot_composer import (
    AS,
    Clause,
    FROM,
    GROUP_BY,
    JOIN,
    SELECT,
    Statement,
    WHERE,
    all_of,
    between,
    count_rows,
    derived,
    equals,
    if_else,
    is_in,
    mapped,
    statement,
    sum_of,
)
from references import ops

EVENT_TYPE_VALUES = ["start", "retry", "fail", "finish"]
TEAM_NAME_BY_TEAM = {
    "data": "Data platform",
    "web": "Web",
}

e = AS(ops.job_events, "e")
ev = derived("ev", statement(
    SELECT(
        e.job_id,
        e.dt,
        e.event_type,
    ),
    FROM(e),
    WHERE(all_of(
        between(e.dt, "2026-09-20", "2026-09-24"),
        is_in(e.event_type, EVENT_TYPE_VALUES),
    )),
))
o = AS(ops.job_owners, "o")
per_team = derived("per_team", statement(
    SELECT(
        o.team,
        ev.dt,
        AS(count_rows(), "events"),
        AS(sum_of(if_else(equals(ev.event_type, "fail"), 1, 0)), "fails"),
    ),
    FROM(ev),
    # CONVERSION: reads_all_partitions=True, as the original did; check it
    JOIN(o, ON=all_of(
        equals(o.job_id, ev.job_id),
        equals(o.dt, ev.dt),
    ), reads_all_partitions=True),
    GROUP_BY(o.team, ev.dt),
))
p = AS(per_team, "p")


def rows() -> list[Clause]:
    """The clauses of the query, which converted() and a write share."""
    return [
        SELECT(
            p.team,
            AS(mapped(p.team, TEAM_NAME_BY_TEAM, otherwise="Other"), "team_name"),
            p.events,
            p.fails,
            p.dt,
        ),
        FROM(p),
    ]


def converted() -> Statement:
    """The query, as the Hive this Conversion was made from writes it."""
    return statement(*rows())


In [ ]:
# Its Hive against the original's, as given in original.hql
ORIGINAL = pathlib.Path("original.hql").read_text(encoding="utf-8")
show_hive(converted())
print(compare_hive(converted(), ORIGINAL))

In [ ]:
# Its rows against the original's, through your query, on one Partition:
# print(compare_with_original(converted(), ORIGINAL, query=query, partition={"dt": "<a value>"}))